# ⚾ 재대결 회피 효과의 이벤트 단위 Mixed-Effects 검증

* **작성자:** DY
* **작성일:** 2026-10-05
* **목적:** 노트북 3·4·6은 재대결 이벤트 27,391건을 투수 439명의 평균으로 뭉개고, 그 평균을 회피율 3분위로
  잘라 147명씩 비교했습니다. 리서치 에이전트가 찾은 대로, 이 압축 과정 자체가 저표본 투수를 양 극단에
  몰아넣는 통계적 함정이었습니다(회피율 최상·최하위 10명 중 8명이 표본(n_events) 중앙값 이하). 이 노트북은
  같은 질문("재대결에서 장타 맞은 구종을 회피하는 것이 유리한가")을, **투수 평균으로 뭉개지 않고 이벤트
  27,391건을 그대로 단위로 삼아, `pitcher`를 랜덤효과로 둔 mixed-effects 모델로 직접 검정**합니다.
  * (D) 연속형: `rvae ~ avoided + ... + (1 | pitcher)` — `lmer`
  * (E) 이항: `allowed_xbh_again ~ avoided + ... + (1 | pitcher)` — `glmer`
  * 투수당 최소 표본 하한을 두지 않습니다 — mixed-effects의 부분 풀링(partial pooling)이 저표본 투수를
    적절히 수축시켜 반영하는 게 이 설계의 핵심입니다.
* **설계 문서:** `docs/superpowers/specs/2026-10-05-rematch-avoidance-mixed-effects-design.md`

In [1]:
# 1. 경로 설정 (src 폴더 접근용)
import sys, os
sys.path.append(os.path.abspath('..'))

# 2. Autoreload 설정
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.analysis.avoidance_stats import compute_season_usage_rate
from src.analysis.glmer_runner import (
    check_convergence,
    extract_fixed_effects_table,
    extract_lmer_fixed_effects,
    extract_variance_components,
    fit_glmer,
    fit_lmer,
)
from src.analysis.mixed_effects_model import compute_icc, compute_icc_gaussian
from src.analysis.run_same_batter_rematch_analysis import build_xbh_rematch
from src.preprocessing.build_next_ab_dataset import EXTRA_BASE_HIT_EVENTS
from src.preprocessing.pitch_family import map_pitch_family
from src.preprocessing.research_sample import KEY_COLUMNS, list_research_csvs

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)

TRAIN_SEASONS = (2021, 2022, 2023, 2024, 2025)
VALID_SEASON = 2026
RANDOM_STATE = 20261003

Error importing in API mode: ImportError("dlopen(/Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib\n  Referenced from: <9F0E20C3-D782-31B5-943E-66F62BCE77E8> /Users/dongyunkwak/GitHub/9th-first-project-baseball-2/.venv/lib/python3.14/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.6/Resources/lib/libRblas.dylib' (no such file)")


Trying to import in ABI mode.


## 1. 팀 CSV 읽기

`data/raw` 없이 팀 research CSV 6개를 읽습니다. `woba_value`/`woba_denom`은 그 타석을 끝낸 결정구에만 채워져 있고
(실측 확인됨), 이게 GBM의 학습·평가 단위(타석이 아니라 결정구 하나)를 정합니다.

In [2]:
LOAD_COLUMNS = [
    'game_pk', 'game_date', 'at_bat_number', 'pitch_number', 'pitcher', 'player_name', 'batter',
    'stand', 'p_throws', 'pitch_type', 'events', 'balls', 'strikes', 'outs_when_up',
    'inning', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_score', 'away_score',
    'woba_value', 'woba_denom',
]


def load_team_csv() -> pd.DataFrame:
    frames = [
        pd.read_csv(path, usecols=LOAD_COLUMNS, encoding='utf-8-sig', low_memory=False)
        for path in list_research_csvs()
    ]
    pitches = pd.concat(frames, ignore_index=True)
    pitches['season'] = pd.to_datetime(pitches['game_date']).dt.year
    return pitches


pitches = load_team_csv()
assert len(pitches) == 3_592_302 and pitches['pitcher'].nunique() == 1_067
assert not pitches.duplicated(KEY_COLUMNS).any()
print(f'{len(pitches):,}행, 투수 {pitches["pitcher"].nunique():,}명')

3,592,302행, 투수 1,067명


## 2. 상황 피처

`score_diff`는 `src/preprocessing/build_next_ab_dataset.py`의 `compute_score_diff()`와 같은 정의(투수팀 관점,
수비팀 − 공격팀)를 벡터화한 것입니다.

In [3]:
SITUATION_COLUMNS = ['runners_n', 'risp', 'score_diff', 'platoon_match']


def add_situation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    on1, on2, on3 = out['on_1b'].notna(), out['on_2b'].notna(), out['on_3b'].notna()
    out['runners_n'] = on1.astype(int) + on2.astype(int) + on3.astype(int)
    out['risp'] = (on2 | on3).astype(int)
    out['score_diff'] = np.where(
        out['inning_topbot'] == 'Top', out['home_score'] - out['away_score'], out['away_score'] - out['home_score']
    )
    out['platoon_match'] = np.where(
        out['stand'].isna() | out['p_throws'].isna(), np.nan, (out['stand'] == out['p_throws']).astype(float)
    )
    return out

In [4]:
_t = pd.DataFrame({
    'on_1b': [np.nan, 1.0, np.nan], 'on_2b': [np.nan, np.nan, 2.0], 'on_3b': [np.nan, np.nan, np.nan],
    'inning_topbot': ['Top', 'Bot', 'Top'], 'home_score': [3, 5, 2], 'away_score': [3, 2, 4],
    'stand': ['L', 'R', 'R'], 'p_throws': ['R', 'R', 'L'],
})
_r = add_situation_columns(_t)
assert _r['runners_n'].tolist() == [0, 1, 1]
assert _r['risp'].tolist() == [0, 0, 1]
assert _r['score_diff'].tolist() == [0, -3, -2]
assert _r['platoon_match'].tolist() == [0.0, 1.0, 0.0]
print('add_situation_columns 테스트 통과')

add_situation_columns 테스트 통과


In [5]:
pitches = add_situation_columns(pitches)
print(pitches[SITUATION_COLUMNS].describe().round(3))

         runners_n         risp   score_diff  platoon_match
count  3592302.000  3592302.000  3592302.000    3592302.000
mean         0.580        0.237        0.189          0.447
std          0.775        0.425        2.936          0.497
min          0.000        0.000      -25.000          0.000
25%          0.000        0.000       -1.000          0.000
50%          0.000        0.000        0.000          0.000
75%          1.000        0.000        2.000          1.000
max          3.000        1.000       24.000          1.000


## 3. 재대결 이벤트

장타 재대결 이벤트는 노트북 01·02·03과 같은 건수(27,391건)가 나와야 합니다. 투수 평균으로 뭉개는 단계가
없으므로, 여기서는 포함 기준(최소 이벤트 수)도 회피율 집계도 하지 않습니다 — 이벤트 자체가 분석 단위입니다.

In [6]:
season_usage = compute_season_usage_rate(pitches)
xbh = build_xbh_rematch(pitches, season_usage)
xbh = xbh[xbh['has_next_ab']].copy()
assert len(xbh) == 27_391, len(xbh)
print(f'재대결 이벤트 {len(xbh):,}건')

2026-10-05 17:30:29,708 [INFO] 장타 70319건 중 같은 타자·같은 투수 재대결 27391건 (39.0%)
        xbh_total  same_batter_rematch  rematch_share
season                                               
2021        12709                 4688         0.3689
2022        12186                 4623         0.3794
2023        13137                 5280         0.4019
2024        12427                 4977         0.4005
2025        12394                 4916         0.3966
2026         7466                 2907         0.3894


재대결 이벤트 27,391건


## 4. GBM 학습 테이블

표본 전체(재대결에 국한하지 않음)의 모든 결정구에서, 그 결정구의 구종이 그 투수·시즌에서 차지하는
`season_usage_rate`를 붙입니다. `pitcher_id`는 피처에 넣지 않습니다.

In [7]:
MODEL_FEATURES = [
    'pitch_type', 'stand', 'p_throws', 'balls', 'strikes', 'outs_when_up', 'runners_n', 'risp', 'score_diff',
    'season_usage_rate',
]
CATEGORICAL_FEATURES = ['pitch_type', 'stand', 'p_throws']


def build_decisive_pitch_table(pitches_sit: pd.DataFrame, season_usage_: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna() & pitches_sit['pitch_type'].notna()].copy()
    decisive = decisive.merge(
        season_usage_[['pitcher', 'season', 'pitch_type', 'season_usage_rate']],
        on=['pitcher', 'season', 'pitch_type'], how='left',
    )
    for col in CATEGORICAL_FEATURES:
        decisive[col] = decisive[col].astype('category')
    return decisive

In [8]:
_px = pd.DataFrame({
    'pitcher': [1, 1, 1, 1], 'season': [2021, 2021, 2021, 2021], 'pitch_type': ['SL', 'FF', 'SL', None],
    'events': [None, None, 'single', 'walk'], 'woba_value': [np.nan, np.nan, 0.9, 0.7], 'woba_denom': [np.nan, np.nan, 1.0, 1.0],
    'outs_when_up': [0, 0, 0, 0], 'balls': [0, 0, 1, 4], 'strikes': [0, 0, 2, 0],
    'runners_n': [0, 0, 0, 0], 'risp': [0, 0, 0, 0], 'score_diff': [0, 0, 0, 0],
    'platoon_match': [1.0, 1.0, 1.0, 1.0], 'stand': ['R', 'R', 'R', 'R'], 'p_throws': ['R', 'R', 'R', 'R'],
})
_su = pd.DataFrame({'pitcher': [1, 1], 'season': [2021, 2021], 'pitch_type': ['SL', 'FF'], 'season_usage_rate': [0.6, 0.4]})
_tbl = build_decisive_pitch_table(_px, _su)
assert len(_tbl) == 1  # 결정구(woba_value 있고 pitch_type도 있는 행)만 1건 -- 4행은 woba_value는 있지만 pitch_type이 없어 제외
assert _tbl.iloc[0]['pitch_type'] == 'SL' and np.isclose(_tbl.iloc[0]['season_usage_rate'], 0.6)
assert _tbl.iloc[0]['balls'] == 1 and _tbl.iloc[0]['strikes'] == 2
assert str(_tbl['pitch_type'].dtype) == 'category' and str(_tbl['stand'].dtype) == 'category'
print('build_decisive_pitch_table 테스트 통과')

build_decisive_pitch_table 테스트 통과


In [9]:
decisive_table = build_decisive_pitch_table(pitches, season_usage)
assert decisive_table['season_usage_rate'].notna().mean() > 0.99, '구사율 매칭 실패 비율이 높음'
print(f'결정구 {len(decisive_table):,}건, season_usage_rate 결측 {decisive_table["season_usage_rate"].isna().sum()}건')

결정구 917,717건, season_usage_rate 결측 0건


## 5. GBM 학습과 검증

2021–2025로 학습, 2026으로 검증합니다. (D) 모델의 타깃인 RVAE는 이 GBM의 기대값 기준입니다.

In [10]:
def fit_run_value_model(table: pd.DataFrame, seasons) -> HistGradientBoostingRegressor:
    train = table[table['season'].isin(seasons)]
    model = HistGradientBoostingRegressor(
        categorical_features='from_dtype', random_state=RANDOM_STATE, max_iter=200, early_stopping=True,
    )
    model.fit(train[MODEL_FEATURES], train['woba_value'])
    return model

In [11]:
train_set = decisive_table[decisive_table['season'].isin(TRAIN_SEASONS)]
valid_set = decisive_table[decisive_table['season'] == VALID_SEASON]
model_cv = fit_run_value_model(decisive_table, TRAIN_SEASONS)
pred_valid = model_cv.predict(valid_set[MODEL_FEATURES])
baseline_mae = mean_absolute_error(valid_set['woba_value'], np.full(len(valid_set), train_set['woba_value'].mean()))
model_mae = mean_absolute_error(valid_set['woba_value'], pred_valid)
model_mse = mean_squared_error(valid_set['woba_value'], pred_valid)
model_r2 = r2_score(valid_set['woba_value'], pred_valid)
print(f'베이스라인(평균 예측) MAE: {baseline_mae:.4f}')
print(f'GBM MAE: {model_mae:.4f} (베이스라인 대비 {(baseline_mae - model_mae) / baseline_mae:.1%}, 참고용)')
print(f'GBM MSE: {model_mse:.4f}, R²: {model_r2:.4f} (평균만 예측하면 R²=0)')
assert model_r2 > 0, 'GBM이 평균만 예측하는 것보다 분산을 못 줄임 -- 피처/학습을 재검토해야 함'

final_model = fit_run_value_model(decisive_table, decisive_table['season'].unique())
print('전체 데이터로 재학습한 최종 모델 준비 완료')

베이스라인(평균 예측) MAE: 0.4322
GBM MAE: 0.3989 (베이스라인 대비 7.7%, 참고용)
GBM MSE: 0.2445, R²: 0.0568 (평균만 예측하면 R²=0)


전체 데이터로 재학습한 최종 모델 준비 완료


In [12]:
def compute_rvae(df: pd.DataFrame, model) -> pd.Series:
    expected = model.predict(df[MODEL_FEATURES])
    return df['woba_value'] - expected

In [13]:
class _StubModel:
    def predict(self, X):
        return np.full(len(X), 0.3)

_df = pd.DataFrame({'woba_value': [0.9, 0.0], **{c: [0] * 2 for c in MODEL_FEATURES}})
_rvae = compute_rvae(_df, _StubModel())
assert np.allclose(_rvae.to_numpy(), [0.6, -0.3])
print('compute_rvae 테스트 통과')

compute_rvae 테스트 통과


## 6. 재대결 결과 테이블 + 모델링 데이터셋

노트북 3의 `find_rematch_decisive_pitch`와 같은 틀이지만, `platoon_match`·`events`를 추가로 들고 오고
`xbh_`의 `reused_same_type`도 같이 넘겨서(그 재대결에서 장타 맞은 구종을 회피했는지), 이후 `avoided`·
`allowed_xbh_again`을 바로 유도할 수 있게 합니다.

In [14]:
def find_rematch_outcome_pitch(xbh_: pd.DataFrame, pitches_sit: pd.DataFrame) -> pd.DataFrame:
    decisive = pitches_sit[pitches_sit['woba_value'].notna()][
        ['game_pk', 'pitcher', 'season', 'at_bat_number', *MODEL_FEATURES, 'platoon_match', 'events', 'woba_value']
    ]
    keys = xbh_[['game_pk', 'pitcher', 'next_at_bat_number', 'reused_same_type']].rename(
        columns={'next_at_bat_number': 'at_bat_number'}
    )
    out = keys.merge(decisive, on=['game_pk', 'pitcher', 'at_bat_number'], how='left', validate='many_to_one')
    return out

In [15]:
_xbh_fix = pd.DataFrame({
    'game_pk': [100], 'pitcher': [10], 'next_at_bat_number': [5.0], 'reused_same_type': [1.0],
})
_pt_fix = pd.DataFrame({
    'game_pk': [100], 'pitcher': [10], 'season': [2021], 'at_bat_number': [5],
    'woba_value': [0.9], 'events': ['double'], 'pitch_type': ['SL'], 'stand': ['R'], 'p_throws': ['R'],
    'balls': [1], 'strikes': [2], 'outs_when_up': [0], 'runners_n': [0], 'risp': [0], 'score_diff': [0],
    'season_usage_rate': [0.5], 'platoon_match': [1.0],
})
_found = find_rematch_outcome_pitch(_xbh_fix, _pt_fix)
assert len(_found) == 1
assert _found.iloc[0]['events'] == 'double' and _found.iloc[0]['platoon_match'] == 1.0
assert _found.iloc[0]['reused_same_type'] == 1.0
print('find_rematch_outcome_pitch 테스트 통과')

find_rematch_outcome_pitch 테스트 통과


In [16]:
REQUIRED_OUTCOME_COLUMNS = sorted(set(['events', 'woba_value', 'platoon_match', *MODEL_FEATURES]))


def build_avoidance_outcome_dataset(found: pd.DataFrame) -> pd.DataFrame:
    out = found.copy()
    out['pitch_family'] = out['pitch_type'].map(map_pitch_family)
    out = out.dropna(subset=[*REQUIRED_OUTCOME_COLUMNS, 'pitch_family', 'reused_same_type'])
    out['avoided'] = (1 - out['reused_same_type']).astype(int)
    out['allowed_xbh_again'] = out['events'].isin(EXTRA_BASE_HIT_EVENTS).astype(int)
    out['pitcher'] = out['pitcher'].astype(str)
    return out

In [17]:
_found2 = pd.DataFrame({
    'pitcher': [10, 20], 'season': [2021, 2021], 'reused_same_type': [1.0, 0.0],
    'pitch_type': ['SL', 'FF'], 'events': ['double', 'field_out'], 'woba_value': [0.9, 0.0],
    'balls': [1, 0], 'strikes': [2, 1], 'outs_when_up': [0, 1], 'score_diff': [0, -1],
    'platoon_match': [1.0, 0.0], 'season_usage_rate': [0.5, 0.4], 'stand': ['R', 'L'], 'p_throws': ['R', 'R'],
    'runners_n': [0, 1], 'risp': [0, 0],
})
_out = build_avoidance_outcome_dataset(_found2)
assert _out['avoided'].tolist() == [0, 1]  # reused -> not avoided; didn't reuse -> avoided
assert _out['allowed_xbh_again'].tolist() == [1, 0]
assert _out['pitch_family'].tolist() == ['breaking', 'fastball']
assert pd.api.types.is_string_dtype(_out['pitcher']) and _out['pitcher'].tolist() == ['10', '20']

_found3 = pd.DataFrame({
    'pitcher': [10], 'season': [2021], 'reused_same_type': [1.0], 'pitch_type': ['SL'],
    'events': [None], 'woba_value': [np.nan], 'balls': [1], 'strikes': [2], 'outs_when_up': [0],
    'score_diff': [0], 'platoon_match': [1.0], 'season_usage_rate': [0.5], 'stand': ['R'], 'p_throws': ['R'],
    'runners_n': [0], 'risp': [0],
})
_out3 = build_avoidance_outcome_dataset(_found3)
assert len(_out3) == 0  # missing events/woba_value row must be dropped, not default allowed_xbh_again=0

_found4 = pd.DataFrame({
    'pitcher': [10, 20], 'season': [2021, 2021], 'reused_same_type': [1.0, 0.0],
    'pitch_type': [None, 'FF'], 'events': ['field_out', 'field_out'], 'woba_value': [0.0, 0.0],
    'balls': [1, 0], 'strikes': [2, 1], 'outs_when_up': [0, 1], 'score_diff': [0, -1],
    'platoon_match': [1.0, 0.0], 'season_usage_rate': [0.5, 0.4], 'stand': ['R', 'L'], 'p_throws': ['R', 'R'],
    'runners_n': [0, 1], 'risp': [0, 0],
})
_out4 = build_avoidance_outcome_dataset(_found4)
assert len(_out4) == 1 and _out4.iloc[0]['pitcher'] == '20'  # pitch_type=None -> pitch_family=None -> dropped
assert _out4['pitch_family'].tolist() == ['fastball']
print('build_avoidance_outcome_dataset 테스트 통과')

build_avoidance_outcome_dataset 테스트 통과


## 7. 실데이터 적용

In [18]:
found = find_rematch_outcome_pitch(xbh, decisive_table)
missing = found['woba_value'].isna().sum()
print(f'재대결 {len(found):,}건 중 결정구를 못 찾은 건수: {missing} ({missing / len(found):.2%})')
model_data = build_avoidance_outcome_dataset(found.dropna(subset=['woba_value']))
model_data['rvae'] = compute_rvae(model_data, final_model)
dropped_total = len(found) - len(model_data)
print(f'모델링에 쓸 재대결: {len(model_data):,}건 (제외 {dropped_total}건), 투수 {model_data["pitcher"].nunique()}명')
print('포함 기준 하한 없음 확인 -- 노트북 3의 투수 439명(재대결 10건 이상)보다 많아야 함:',
      model_data['pitcher'].nunique() > 439)
display(model_data[['avoided', 'rvae', 'allowed_xbh_again']].describe())

재대결 27,391건 중 결정구를 못 찾은 건수: 135 (0.49%)
모델링에 쓸 재대결: 27,245건 (제외 146건), 투수 695명
포함 기준 하한 없음 확인 -- 노트북 3의 투수 439명(재대결 10건 이상)보다 많아야 함: True


,avoided,rvae,allowed_xbh_again
count,27245.000000,27245.000000,27245.000000
mean,0.543402,0.022022,0.089484
std,0.498122,0.523265,0.285447
min,0.000000,-0.678765,0.000000
25%,0.000000,-0.375685,0.000000
50%,1.000000,-0.193003,0.000000
75%,1.000000,0.336228,0.000000
max,1.000000,1.840325,1.000000


## 8. 모델 (D): 연속형 RVAE

`rvae ~ avoided + balls + strikes + outs_when_up + score_diff + platoon_match + pitch_family + factor(season)
+ (1 | pitcher)` — `lmer`(가우시안). `avoided`의 추정치가 핵심입니다(음수면 회피가 RVAE를 낮춰 투수에게
유리하다는 뜻).

In [19]:
FORMULA_D = (
    'rvae ~ avoided + balls + strikes + outs_when_up + score_diff + platoon_match '
    '+ pitch_family + factor(season) + (1 | pitcher)'
)
fit_lmer(model_data, FORMULA_D)
print('=== (D) 수렴 상태 ===')
print(check_convergence())
fixed_d = extract_lmer_fixed_effects()
print('=== (D) 고정효과 (avoided 포함) ===')
display(fixed_d.round(4))
vc_d = extract_variance_components('pitcher')
print('(D) 분산 성분:', vc_d)
if vc_d['pitcher_intercept'] is None or vc_d['residual'] is None:
    print('(D) ICC 계산 불가 -- 분산 성분 중 하나가 None (위 출력 참고)')
    icc_d = None
else:
    icc_d = compute_icc_gaussian(vc_d['pitcher_intercept'], vc_d['residual'])
    print(f'(D) ICC (투수 간 분산 비율): {icc_d:.4f} -- 노트북 3의 ω²=0.266과 비교 가능')

2026-10-05 17:30:59,485 [WARNING] R callback write-console: Loading required package: Matrix
  


=== (D) 수렴 상태 ===
OK: no convergence warnings; isSingular=False
=== (D) 고정효과 (avoided 포함) ===


,term,estimate,std_error,t_value,ci_low,ci_high,p_value
0,(Intercept),0.0351,0.0132,2.6695,0.0093,0.0609,0.0076
1,avoided,-0.0056,0.0067,-0.8296,-0.0187,0.0076,0.4068
2,balls,0.0018,0.0031,0.5862,-0.0043,0.0080,0.5578
3,strikes,-0.0026,0.0043,-0.6015,-0.0110,0.0058,0.5475
4,outs_when_up,-0.0034,0.0039,-0.8772,-0.0110,0.0042,0.3804
5,score_diff,-0.0006,0.0011,-0.5473,-0.0027,0.0015,0.5842
6,platoon_match,-0.0000,0.0066,-0.0068,-0.0130,0.0129,0.9946
7,pitch_familyfastball,-0.0015,0.0074,-0.1982,-0.0160,0.0131,0.8429
8,pitch_familyoffspeed,-0.0158,0.0096,-1.6395,-0.0346,0.0031,0.1011
9,factor(season)2022,0.0017,0.0109,0.1578,-0.0197,0.0231,0.8746


(D) 분산 성분: {'pitcher_intercept': 0.0005366390536871161, 'residual': 0.27331855875139927}
(D) ICC (투수 간 분산 비율): 0.0020 -- 노트북 3의 ω²=0.266과 비교 가능


## 9. 모델 (E): 또-장타 (이항) + ICC 비교

`allowed_xbh_again ~ avoided + balls + strikes + outs_when_up + score_diff + platoon_match + pitch_family
+ factor(season) + (1 | pitcher)` — `glmer`(이항). (D)와 같은 고정효과 구조라 ICC를 나란히 비교할 수 있습니다.

In [20]:
FORMULA_E = (
    'allowed_xbh_again ~ avoided + balls + strikes + outs_when_up + score_diff + platoon_match '
    '+ pitch_family + factor(season) + (1 | pitcher)'
)
fit_glmer(model_data, FORMULA_E)
print('=== (E) 수렴 상태 ===')
print(check_convergence())
fixed_e = extract_fixed_effects_table()
print('=== (E) 고정효과 (avoided 포함, odds_ratio 포함) ===')
display(fixed_e.round(4))
vc_e = extract_variance_components('pitcher')
print('(E) 분산 성분:', vc_e)
if vc_e['pitcher_intercept'] is None:
    print('(E) ICC 계산 불가 -- 분산 성분이 None (위 출력 참고)')
    icc_e = None
else:
    icc_e = compute_icc(vc_e['pitcher_intercept'])
    print(f'(E) ICC (투수 간 분산 비율): {icc_e:.4f}')

print('\n=== ICC 비교 (투수 간 분산이 설명하는 비율) ===')
print(f'노트북 3 ω² (회피율, 편향 보정): 0.266')
print(f'(D) RVAE 모델 ICC: {icc_d if icc_d is None else f"{icc_d:.4f}"}')
print(f'(E) 또-장타 모델 ICC: {icc_e if icc_e is None else f"{icc_e:.4f}"}')

=== (E) 수렴 상태 ===
OK: no convergence warnings; isSingular=False


=== (E) 고정효과 (avoided 포함, odds_ratio 포함) ===


,term,estimate,std_error,z_value,p_value,odds_ratio,or_ci_low,or_ci_high
0,(Intercept),-1.5367,0.0848,-18.1157,0.0000,0.2151,0.1822,0.2540
1,avoided,0.0126,0.0460,0.2738,0.7842,1.0127,0.9254,1.1082
2,balls,-0.0803,0.0218,-3.6907,0.0002,0.9229,0.8843,0.9631
3,strikes,-0.4085,0.0279,-14.6536,0.0000,0.6647,0.6293,0.7020
4,outs_when_up,-0.0579,0.0263,-2.2029,0.0276,0.9438,0.8964,0.9936
5,score_diff,-0.0102,0.0073,-1.3971,0.1624,0.9898,0.9758,1.0041
6,platoon_match,-0.0732,0.0447,-1.6388,0.1012,0.9294,0.8514,1.0145
7,pitch_familyfastball,-0.1211,0.0492,-2.4584,0.0140,0.8860,0.8045,0.9758
8,pitch_familyoffspeed,-0.2284,0.0666,-3.4284,0.0006,0.7958,0.6984,0.9068
9,factor(season)2022,0.0235,0.0724,0.3249,0.7453,1.0238,0.8883,1.1800


(E) 분산 성분: {'pitcher_intercept': 0.0095268207106703, 'residual': None}
(E) ICC (투수 간 분산 비율): 0.0029

=== ICC 비교 (투수 간 분산이 설명하는 비율) ===
노트북 3 ω² (회피율, 편향 보정): 0.266
(D) RVAE 모델 ICC: 0.0020
(E) 또-장타 모델 ICC: 0.0029


## 10. 해석

* **표본.** 재대결 27,391건 중 135건(0.49%, 노트북 3과 같은 원인 — 대부분 고의사구로 인한 구종 미분류)은
  결정구를 못 찾았고, 추가로 11건이 다른 필수 컬럼 결측으로 빠져 최종 27,245건(투수 695명)을 모델링에
  썼습니다. 포함 기준 하한을 전혀 두지 않았는데도(노트북 3은 재대결 10건 이상인 439명만 포함) 투수 수가
  오히려 더 많습니다 — 이 노트북의 핵심 설계(저표본 투수도 mixed-effects의 부분 풀링으로 자연스럽게 반영)가
  의도대로 작동했다는 뜻입니다.
* **수렴.** (D), (E) 둘 다 `isSingular=False`, 수렴 경고 없음(`check_convergence()` 출력 그대로). 추가
  랜덤효과 구조(랜덤 슬롭 등)로 확장할 필요 없이 `(1 | pitcher)` 단순 구조로 안정적으로 적합됐습니다.
* **(D) 연속형 RVAE.** `avoided` 추정치 −0.0056 [95% CI −0.0187, 0.0076], p=0.407 — 노트북 3과 같은 방향
  (회피가 RVAE를 낮춰 투수에게 유리한 쪽)이지만 0과 통계적으로 구별되지 않습니다. 신뢰구간이 0을 한참
  여유 있게 포함합니다.
* **(E) 또-장타 (이항).** `avoided` 추정치 0.0126(승산비 1.013) [95% CI 0.925, 1.108], p=0.784 — 방향조차
  일관되지 않고(승산비가 1에 극히 가까움), 역시 유의하지 않습니다.
* **ICC 비교 — 가장 중요한 발견.** 노트북 3의 ω²(회피율, 편향 보정) = 0.266이었던 것과 달리, 이 노트북의
  ICC는 (D) 0.0020, (E) 0.0029로 **100배 넘게 작습니다.** 이 둘은 서로 다른 것을 재는 수치이긴 합니다 —
  ω²는 "회피율(투수의 성향)" 자체의 투수 간 분산 비율이고, 여기 ICC는 "RVAE/또-장타(결과)"의 투수 간 분산
  비율입니다. 하지만 방향은 분명합니다: 투수 평균으로 뭉개서 비교했을 때 보이던 "투수마다 차이가 크다"는
  인상은, 상황(볼카운트·아웃·주자·점수차·투타 상성)과 구종군·시즌을 고정효과로 제대로 통제하고 나면 결과
  쪽에서는 거의 사라집니다 — 즉 에이전트 2가 진단한 대로, 3분위로 자르고 투수 평균을 비교하던 방식이
  저표본 투수를 극단에 몰아넣어 "투수차"처럼 보이게 만들었을 가능성이 결과 쪽에서도 뒷받침됩니다.
* **결론 (유의하지 않음, 그리고 왜).** 이벤트 단위로, 뭉개지 않고, 투수를 랜덤효과로 제대로 둔 가장
  엄밀한 검정에서도 "회피가 재대결 결과를 유의하게 개선한다"는 증거는 나오지 않았습니다. 이게 방법이
  허술해서가 아닙니다 — 오히려 지금까지 쓴 어떤 방법보다 통계적으로 정직한 검정이고, 그 결과가 노트북
  3·4·6의 경계선 p값(0.06~0.25)보다 더 분명하게 "차이 없음" 쪽으로 수렴했습니다. ICC가 둘 다 0에 가깝다는
  건 "그 투수가 누구인지"가 재대결 결과를 거의 설명하지 못한다는 뜻이고, 이는 "회피 여부"라는 투수 수준
  행동 하나로 결과를 예측하려는 시도 자체의 한계를 보여줍니다 — 재대결 결과는 압도적으로 상황·실행·운에
  좌우되고, 회피라는 단일 결정이 거기 보탤 수 있는 몫은 이 데이터로 검출 가능한 범위 안에서는 없었습니다.